# Erdős–Straus Conjecture Solver — SageMaker Studio Lab

**Guinea Pig Trench LLC**

---

$$\frac{4}{n} = \frac{1}{x} + \frac{1}{y} + \frac{1}{z}$$

Verified to **100,000,000**. This notebook pushes the frontier.

SageMaker Studio Lab: 4 vCPUs, 16 GB RAM, 12-hour sessions, **persistent storage**.

GitHub: [Commencethescourge/erdos-straus-solver](https://github.com/Commencethescourge/erdos-straus-solver)

In [ ]:
import os, math, multiprocessing as mp, csv, time

cpu_count = mp.cpu_count()
workers = max(2, cpu_count)
print(f'CPUs: {cpu_count} | Workers: {workers}')

# SageMaker Studio Lab persistent storage
RESULTS_DIR = os.path.expanduser('~/erdos_straus')
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'Results dir: {RESULTS_DIR} (persists across sessions)')

In [ ]:
def solve_single(n, step_cap=20_000_000, y_cap_per_x=2_000_000):
    if n <= 1:
        return None
    steps = 0
    x_min = max(1, math.ceil(n / 4))
    x_max = n
    for x in range(x_min, x_max + 1):
        num_r = 4 * x - n
        if num_r <= 0:
            steps += 1
            if steps >= step_cap:
                return None
            continue
        den_r = n * x
        y_min = math.ceil(den_r / num_r)
        y_max = 2 * den_r // num_r
        y_steps = 0
        for y in range(max(x, y_min), y_max + 1):
            steps += 1
            y_steps += 1
            if steps >= step_cap:
                return None
            if y_steps >= y_cap_per_x:
                break
            denom_z = num_r * y - den_r
            if denom_z <= 0:
                continue
            num_z = den_r * y
            if num_z % denom_z == 0:
                z = num_z // denom_z
                if z >= y:
                    return {'n': n, 'x': x, 'y': y, 'z': z, 'steps': steps}
    return None

def worker(args):
    n, cap = args
    r = solve_single(n, step_cap=cap)
    if r:
        r['solved'] = True
        return r
    return {'n': n, 'x': 0, 'y': 0, 'z': 0, 'steps': cap, 'solved': False}

t = solve_single(5)
assert t and t['z'] == 20
print('Solver loaded.')

In [ ]:
# === Configure range ===
n_start = 100_000_001
n_end   = 101_000_000
step_cap = 20_000_000

ns = [n for n in range(n_start, n_end + 1) if n % 24 in (1, 17)]
out_file = os.path.join(RESULTS_DIR, f'results_{n_start}_{n_end}.csv')

# Auto-resume
done = set()
if os.path.exists(out_file):
    with open(out_file) as f:
        for row in csv.DictReader(f):
            done.add(int(row['n']))
    print(f'Resuming: {len(done):,} already done')

remaining = [n for n in ns if n not in done]
print(f'Targets: {len(ns):,} total | {len(remaining):,} remaining')

In [ ]:
# === Solve ===
if not remaining:
    print('All done!')
else:
    fields = ['n', 'x', 'y', 'z', 'steps', 'solved']
    total = len(remaining)
    batch_size = workers * 8
    solved_count = len(done)
    unsolved_count = 0
    all_results = []
    mode = 'a' if done else 'w'
    t0 = time.time()

    print(f'Solving {total:,} | workers={workers} | step_cap={step_cap:,}')

    with open(out_file, mode, newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        if not done:
            writer.writeheader()
        with mp.Pool(workers) as pool:
            for i in range(0, total, batch_size):
                batch = remaining[i:i + batch_size]
                results = pool.map(worker, [(n, step_cap) for n in batch])
                for r in results:
                    writer.writerow(r)
                    all_results.append(r)
                    if r['solved']: solved_count += 1
                    else: unsolved_count += 1
                processed = i + len(batch)
                if processed % (batch_size * 50) < batch_size or processed >= total:
                    f.flush()
                    elapsed = time.time() - t0
                    rate = processed / elapsed
                    eta = (total - processed) / rate if rate > 0 else 0
                    print(f'  [{100*processed/total:5.1f}%] {processed:,}/{total:,} | '
                          f'{rate:.0f}/s | ETA: {eta/60:.0f}m')

    print(f'\nDone in {(time.time()-t0)/60:.1f}m | Solved: {solved_count:,} | Unsolved: {unsolved_count}')
    print(f'Saved: {out_file}')

In [ ]:
# === Stats ===
if 'all_results' in dir() and all_results:
    solved = [r for r in all_results if r['solved']]
    unsolved = [r for r in all_results if not r['solved']]
    print(f'Solved: {len(solved):,} | Unsolved: {len(unsolved):,}')
    if solved:
        best = max(solved, key=lambda r: r['z'])
        print(f'Largest z: {best["z"]:,} (n={best["n"]:,})')
    if unsolved:
        print('Unsolved:')
        for r in unsolved[:20]:
            print(f'  n = {r["n"]:,}')
    else:
        print('Conjecture holds for this range!')